## 1. Tải dữ liệu

Bộ dữ liệu chính của bài toán dự báo là `data/model/features_selected.csv`. Mỗi dòng là **một ngân hàng trong một quý** (22 NH × 34 quý, 2018-Q1 → 2026-Q2), gồm 88 đặc trưng đã qua làm sạch và chọn lọc. Các file còn lại được kiểm nhanh ở Lần 3.

| File | Dùng cho |
|---|---|
| `data/model/features_selected.csv` | Mô hình dự báo 6 chỉ tiêu (NIM, COF, NPL, CASA, LDR, ROE) |
| `data/model/bank_panel_quarterly.csv` | Panel sạch, trước khi tạo đặc trưng |
| `data/model/features_ranking_latest.csv` | Xếp hạng: ảnh chụp số gần nhất của 22 NH |
| `data/interest_rates/deposit_24hmoney_daily.csv` | Lãi suất tiết kiệm theo ngày |

Review app (532.007 dòng, bài toán phân loại) có notebook riêng: `02_EDA_so_bo_review.ipynb`.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Tìm thư mục gốc dự án (chứa data/model) — chạy được từ EDA/, từ gốc dự án, hay trên Colab sau khi tải thư mục lên
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "model").exists())
sys.path.insert(0, str(ROOT / "EDA"))
from eda_utils import read_and_explore_data, quick_profile

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)

DATA = ROOT / "data"
PANEL = DATA / "model" / "features_selected.csv"
print("Thư mục dự án:", ROOT)

Thư mục dự án: D:\KLTN


## 2. Hàm `read_and_explore_data`

Hàm nằm trong `EDA/eda_utils.py`.

1. **Kiểm tra file tồn tại**: không có thì dừng ngay, để khỏi chạy nửa buổi mới phát hiện sai đường dẫn.
2. **Đọc theo phần mở rộng** (CSV, Excel, Parquet, JSON), rồi "chào hỏi" dữ liệu bằng `df.shape` và `df.head()`.
3. **`df.info()`**: số ô không thiếu của từng cột và kiểu dữ liệu có hợp lý không.
4. **`df.describe()`** cho cột số (min, max, độ lệch chuẩn, tứ phân vị) và `describe(include="object")` cho cột phân loại (số hạng mục khác nhau).
5. **Xóa cột không cần** nếu có truyền `drop_columns`. Chỉ xóa những cột thật sự tồn tại, rồi báo lại đã xóa cột nào.
6. **Trùng lặp**: `df.duplicated().sum()`. Nếu bật `auto_drop_duplicates` thì xóa và reset index.
7. **Giá trị thiếu**: `df.isnull().sum()`. Nếu bật `auto_drop_missing` thì xóa dòng.
8. **Phân phối nhãn** khi có `target`: đếm số lượng và phần trăm để xem nhãn có lệch không.



In [2]:
import inspect
print(inspect.getsource(read_and_explore_data))

def read_and_explore_data(
    file_path: str,
    encoding: str = "utf-8-sig",
    sep: str = ",",
    drop_columns: list | None = None,
    target: str | None = None,
    auto_drop_missing: bool = False,
    auto_drop_duplicates: bool = False,
    show: bool = True,
    key_columns: list | None = None,
    pattern: str = "*.csv",
) -> pd.DataFrame:
    """Đọc và khám phá sơ bộ dữ liệu (bản dùng cho notebook, VS Code hoặc Colab).

    Thực hiện:
    - Đọc dữ liệu từ CSV / Excel / Parquet / JSON (một file, hoặc cả thư mục theo `pattern`)
    - Hiển thị thông tin tổng quan, thống kê mô tả, nghi vấn sai kiểu
    - Kiểm tra dữ liệu trùng lặp (cả dòng và khóa), thiếu, phân phối nhãn (nếu có)
    - Loại bỏ cột không cần thiết nếu được chỉ định
    """
    # 1️⃣ Kiểm tra file tồn tại
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ Không tìm thấy file: {file_path}")

    # 2️⃣ Xác định định dạng và đọc dữ liệu
    if os.path.isdir(file_path):
        files = sorted(g

## 3. Run

### Lần 1: dữ liệu

Chỉ truyền đường dẫn và khóa `(ticker, period)`. Chưa truyền mục tiêu.

In [3]:
df = read_and_explore_data(PANEL, key_columns=["ticker", "period"])

✅ Đọc dữ liệu thành công: 748 dòng, 91 cột  (D:\KLTN\data\model\features_selected.csv)


,period,period_end,ticker,loans_g2_percent,npl_percent,equity_to_assets_pct,casa_chg_qoq_pp,cof_chg_qoq_pp,ldr_chg_qoq_pp,nim_chg_qoq_pp,npl_chg_qoq_pp,roe_chg_qoq_pp,cof_ttm_percent,deposit_cost_implied_ttm_percent,loan_yield_ttm_pct,nim_ttm_percent,roa_ttm_percent,roe_ttm_percent,loans_qoq_percent,loans_yoy_percent,...,prog_hatang_500k,prog_lts_15k,prog_nlts_100k,prog_nlts_185k,prog_noxh_145k,prog_noxh_u35,ci_bond_to_assets_pct,corp_bond_to_assets_pct,gov_bond_share_invsec_pct,invsec_to_assets_pct,beta,dividend_yield,excess_ret_q_pct,pb,stock_ret_q_pct,stock_vol_q_pct,cpi_yoy_q_pct,gold_index_yoy_q_pct,usd_index_yoy_q_pct,vnindex_vol_q_pct
0,2018-Q1,2018-03-31,ACB,0.2970,0.6982,5.728519,NaN,NaN,NaN,NaN,NaN,NaN,1.1600,NaN,NaN,3.3525,0.99,17.67,NaN,NaN,...,0,0,0,0,0,0,0.000000,0.000000,99.354074,15.914502,1.40,0.00,19.920725,2.68,39.247312,34.601828,2.82,6.24,-0.023333,24.235096
1,2018-Q1,2018-03-31,BAB,0.0721,0.7687,7.508488,NaN,NaN,NaN,NaN,NaN,NaN,6.7120,NaN,NaN,2.1597,0.76,10.57,NaN,NaN,...,0,0,0,0,0,0,7.946676,0.020653,57.838444,19.457856,NaN,0.00,-24.650092,1.74,-5.323505,41.695938,2.82,6.24,-0.023333,24.235096
2,2018-Q1,2018-03-31,BID,3.1837,1.6170,4.140637,NaN,NaN,NaN,NaN,NaN,NaN,4.6305,NaN,NaN,2.9773,0.61,14.64,NaN,NaN,...,0,0,0,0,0,0,0.000000,0.000000,NaN,11.788546,1.35,0.02,50.836583,2.92,70.163170,56.841531,2.82,6.24,-0.023333,24.235096
3,2018-Q1,2018-03-31,BVB,NaN,NaN,9.217322,NaN,NaN,NaN,NaN,NaN,NaN,1.3100,NaN,NaN,2.2000,0.28,3.10,NaN,NaN,...,0,0,0,0,0,0,NaN,NaN,NaN,17.476578,NaN,0.00,NaN,NaN,NaN,NaN,2.82,6.24,-0.023333,24.235096
4,2018-Q1,2018-03-31,CTG,0.4473,1.2459,5.939520,NaN,NaN,NaN,NaN,NaN,NaN,4.4293,NaN,NaN,2.7419,0.75,12.57,NaN,NaN,...,0,0,0,0,0,0,2.317928,3.827505,41.805555,10.632570,1.64,0.02,23.622131,1.95,42.948718,51.415415,2.82,6.24,-0.023333,24.235096


📋 Thông tin tổng quan:
----------------------------------------------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 748 entries, 0 to 747
Data columns (total 91 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   period                             748 non-null    object 
 1   period_end                         748 non-null    object 
 2   ticker                             748 non-null    object 
 3   loans_g2_percent                   737 non-null    float64
 4   npl_percent                        737 non-null    float64
 5   equity_to_assets_pct               748 non-null    float64
 6   casa_chg_qoq_pp                    714 non-null    float64
 7   cof_chg_qoq_pp                     712 non-null    float64
 8   ldr_chg_qoq_pp                     726 non-null    float64
 9   nim_chg_qoq_pp                     715 non-null    float64
 10

,count,mean,std,min,25%,50%,75%,max
loans_g2_percent,737.0,1.740340,1.517392,0.055900,0.703800,1.308100,2.260600,8.304500
npl_percent,737.0,2.402324,3.042685,0.356800,1.321200,1.856300,2.660500,35.278500
equity_to_assets_pct,748.0,8.716217,2.863385,4.108848,6.491756,8.042531,10.293206,17.587817
casa_chg_qoq_pp,714.0,0.048504,2.050013,-9.115400,-0.928375,0.120500,1.048450,8.900800
cof_chg_qoq_pp,712.0,0.012241,0.479159,-2.559600,-0.200175,0.000000,0.176050,3.640700
ldr_chg_qoq_pp,726.0,0.719322,4.422386,-22.608200,-1.770650,0.647300,3.164850,20.895700
nim_chg_qoq_pp,715.0,-0.008235,0.234889,-2.790100,-0.105950,-0.010000,0.099650,2.660800
npl_chg_qoq_pp,714.0,0.018840,0.801786,-11.072400,-0.166050,0.008200,0.186775,7.316100
roe_chg_qoq_pp,715.0,0.034853,4.779371,-82.900000,-0.780000,-0.020000,0.935000,84.700000
cof_ttm_percent,735.0,4.596077,1.309956,1.160000,3.706600,4.611800,5.397750,8.705200


----------------------------------------------------------------------------------------------------
🔠 Thống kê mô tả (cột phân loại):


,count,unique,top,freq
period,748,34,2018-Q1,22
period_end,748,34,2018-03-31,22
ticker,748,22,ACB,34


⚠️ Cột có thể đang sai kiểu dữ liệu:


,cột,nghi vấn,căn cứ
0,period_end,có thể là NGÀY GIỜ,100% giá trị đổi được sang datetime


----------------------------------------------------------------------------------------------------
✅ Không có dòng trùng lặp.
✅ Khóa ['ticker', 'period'] là duy nhất.
----------------------------------------------------------------------------------------------------
⚠️ 57 cột có giá trị thiếu (tổng 3,782 ô, 5.6% toàn bảng):


,số ô thiếu,tỷ lệ
ind_share_C_asof_pct,210,0.2807
gold_index_yoy_q_pct,198,0.2647
ind_share_F_asof_pct,197,0.2634
car_asof_percent,177,0.2366
app_reviews_log,165,0.2206
toi_yoy_sym_pct,154,0.2059
npat_yoy_sym_pct,154,0.2059
staff_cost_share_pct,132,0.1765
paper_interest_share_pct,113,0.1511
deposit_insurance_to_deposits_pct,98,0.1310


----------------------------------------------------------------------------------------------------
✅ HOÀN TẤT KHÁM PHÁ DỮ LIỆU


### Lần 2: mục tiêu



In [4]:
TARGETS = {"nim_ttm_percent": "NIM", "cof_ttm_percent": "COF", "npl_percent": "NPL",
           "casa_percent": "CASA", "ldr_computed_percent": "LDR", "roe_ttm_percent": "ROE"}
for col, label in TARGETS.items():
    print(f"\n########## {label} ({col})")
    read_and_explore_data(PANEL, target=col, show=False)


########## NIM (nim_ttm_percent)
📊 'nim_ttm_percent' là biến liên tục (hồi quy), 709 giá trị khác nhau:
count    738.0000
mean       3.3690
std        1.3468
min       -1.8992
25%        2.5012
50%        3.1474
75%        3.9600
max        9.4261
Name: nim_ttm_percent, dtype: float64
   Độ lệch (skew): 1.43 · độ nhọn (kurtosis): 5.08 · thiếu: 10

########## COF (cof_ttm_percent)
📊 'cof_ttm_percent' là biến liên tục (hồi quy), 697 giá trị khác nhau:
count    735.0000
mean       4.5961
std        1.3100
min        1.1600
25%        3.7066
50%        4.6118
75%        5.3978
max        8.7052
Name: cof_ttm_percent, dtype: float64
   Độ lệch (skew): 0.06 · độ nhọn (kurtosis): -0.01 · thiếu: 13

########## NPL (npl_percent)
📊 'npl_percent' là biến liên tục (hồi quy), 728 giá trị khác nhau:
count    737.0000
mean       2.4023
std        3.0427
min        0.3568
25%        1.3212
50%        1.8563
75%        2.6605
max       35.2785
Name: npl_percent, dtype: float64
   Độ lệch (skew): 7.09 

Gom lại thành một bảng để so sánh 6 mục tiêu, kèm ngân hàng giữ giá trị lớn nhất và nhỏ nhất:

In [5]:
summary = pd.DataFrame({
    label: {
        "thiếu": int(df[col].isna().sum()),
        "min": df[col].min(), "NH có min": df.loc[df[col].idxmin(), "ticker"],
        "trung vị": df[col].median(),
        "max": df[col].max(), "NH có max": df.loc[df[col].idxmax(), "ticker"],
        "độ lệch": df[col].skew(), "độ nhọn": df[col].kurt(),
    }
    for col, label in TARGETS.items()
}).T
summary.round(2)

,thiếu,min,NH có min,trung vị,max,NH có max,độ lệch,độ nhọn
NIM,10,-1.8992,NVB,3.14735,9.4261,VPB,1.426422,5.075197
COF,13,1.16,ACB,4.6118,8.7052,BAB,0.062206,-0.013464
NPL,11,0.3568,TCB,1.8563,35.2785,NVB,7.090414,58.781818
CASA,11,0.9905,BAB,14.3315,46.977,TCB,0.916952,0.389862
LDR,0,55.289,NVB,99.42055,155.55,VPB,0.116318,0.527791
ROE,10,-92.29,NVB,16.4,32.89,VIB,-4.921959,44.607849


### Lần 3: `show=False` cho các file còn lại



In [6]:
profile = quick_profile(
    {
        "Panel đặc trưng (features_selected)": PANEL,
        "Panel sạch (bank_panel_quarterly)": DATA / "model" / "bank_panel_quarterly.csv",
        "Ảnh chụp xếp hạng (features_ranking_latest)": DATA / "model" / "features_ranking_latest.csv",
        "Lãi suất theo ngày (deposit_24hmoney_daily)": DATA / "interest_rates" / "deposit_24hmoney_daily.csv",
    },
    key_columns={
        "Panel đặc trưng (features_selected)": ["ticker", "period"],
        "Panel sạch (bank_panel_quarterly)": ["ticker", "period"],
        "Ảnh chụp xếp hạng (features_ranking_latest)": ["ticker"],
    },
)
profile

,bộ dữ liệu,dòng,cột,cột số,cột phân loại,dòng trùng,trùng khóa,ô thiếu (%),cột sai kiểu?
0,Panel đặc trưng (features_selected),748,91,88,3,0,0.0,5.56,period_end
1,Panel sạch (bank_panel_quarterly),748,42,31,11,0,0.0,5.63,"period_end, built_at_utc"
2,Ảnh chụp xếp hạng (features_ranking_latest),22,90,87,3,0,0.0,0.61,—
3,Lãi suất theo ngày (deposit_24hmoney_daily),21584,12,2,10,0,NaN,6.84,"date, effective_date, retrieved_at_utc"


In [7]:
rates = pd.read_csv(DATA / "interest_rates" / "deposit_24hmoney_daily.csv", encoding="utf-8-sig")
print(f"Lãi suất: {len(rates):,} dòng, {rates.date.nunique()} ngày ({rates.date.min()} → {rates.date.max()})")
print(f"Dòng không có ticker: {rates.ticker.isna().sum():,} "
      f"({rates.ticker.isna().mean():.0%}) — thuộc {rates.loc[rates.ticker.isna(), 'bank_24hmoney'].nunique()} NH ngoài 22 mã của đề tài")
ranking = pd.read_csv(DATA / "model" / "features_ranking_latest.csv", encoding="utf-8-sig")
print("Ô thiếu trong ảnh chụp xếp hạng:",
      {c: ranking.loc[ranking[c].isna(), "ticker"].tolist() for c in ranking.columns if ranking[c].isna().any()})

Lãi suất: 21,584 dòng, 46 ngày (2023-01-27 → 2026-09-23)
Dòng không có ticker: 9,018 (42%) — thuộc 33 NH ngoài 22 mã của đề tài
Ô thiếu trong ảnh chụp xếp hạng: {'car_asof_percent': ['NVB'], 'deposit_insurance_to_deposits_pct': ['NAB'], 'staff_cost_share_pct': ['SHB'], 'ind_share_C_asof_pct': ['BVB', 'LPB', 'PGB'], 'ind_share_F_asof_pct': ['BVB', 'LPB', 'PGB'], 'app_mean_score': ['BAB'], 'app_pct_1star': ['BAB'], 'app_reply_rate_pct': ['BAB']}


## 4. Ý nghĩa đặc trưng theo nhóm
Ở đây 88 đặc trưng được chia theo khung **CAMELS** cùng một số nhóm bổ sung.

Ký hiệu trong cột *Ghi chú điều tra*:
- 🎯 mục tiêu dự báo
- ★ đi cùng thay đổi một mục tiêu ở quý sau (cùng dấu ở cả hai nửa giai đoạn)
- ⚠️ vấn đề chất lượng: ngoại lai, thiếu nhiều, hoặc rò rỉ cùng quý
- 🔍 giả thuyết cần kiểm ở EDA chi tiết


Bảng dưới đây sinh từ `EDA/tu_dien.py` . Bản đầy đủ ở `EDA/tu_dien_dac_trung.csv`.

In [8]:
from tu_dien import build
tu_dien = build(ROOT)
tu_dien.groupby("nhóm").agg(số_biến=("cột", "size"),
                            mục_tiêu=("ký hiệu", lambda s: s.fillna("").str.contains("🎯").sum()),
                            có_liên_hệ=("ký hiệu", lambda s: s.fillna("").str.contains("★").sum()),
                            có_vấn_đề=("ký hiệu", lambda s: s.fillna("").str.contains("⚠️").sum()))

,số_biến,mục_tiêu,có_liên_hệ,có_vấn_đề
nhóm,,,,
A Chất lượng tài sản,6,1,1,2
C Vốn,4,0,1,2
D Động lượng,6,0,6,6
E Thu nhập & chi phí vốn,12,3,0,4
G Quy mô & tăng trưởng,9,0,3,0
K Khách hàng số,1,0,0,1
L Thanh khoản,13,2,1,3
M Quản trị,6,0,0,2
P Chính sách & cơ cấu tín dụng,11,0,0,3


### 4.1. Sức khỏe ngân hàng (CAMELS)

Nhóm quyết định ngân hàng có an toàn, sinh lời, đủ thanh khoản hay không. Cũng là nơi chứa 6 mục tiêu dự báo.

#### C Vốn

| Cột | Ý nghĩa | Ghi chú điều tra |
|---|---|---|
| **equity_to_assets_pct** ★ | Vốn chủ / tổng tài sản (%) | Đi cùng thay đổi ROE quý sau (ρ = −0,21, cùng dấu ở hai nửa giai đoạn). |
| **car_asof_percent** ⚠️ | CAR hợp nhất gần nhất đã công bố (%) | Số năm gán cho các quý sau (as-of): đổi theo bậc mỗi năm. Thiếu 13% ô. Nhiều NH chỉ công bố CAR theo năm hoặc nửa năm; NVB không có số sau 2024-Q1. |
| **dividend_payout_ttm_pct** ⚠️ 🔍 | Cổ tức tiền mặt đã trả TTM / LNST TTM (%) | Ngoại lai 8,8% ô, 47% là của BAB. Lệch mạnh (skew 3,8): cần biến đổi nếu dùng mô hình tuyến tính. |
| **retained_to_equity_pct** 🔍 | LN chưa phân phối / vốn chủ (%) | Lệch mạnh (skew −2,4): cần biến đổi nếu dùng mô hình tuyến tính. |

#### A Chất lượng tài sản

| Cột | Ý nghĩa | Ghi chú điều tra |
|---|---|---|
| **loans_g2_percent** ★ | Nợ nhóm 2 / dư nợ (%) | Đi cùng thay đổi ROE quý sau (ρ = −0,21, cùng dấu ở hai nửa giai đoạn). |
| **npl_percent** 🎯 ⚠️ 🔍 | Nợ xấu / dư nợ (thuyết minh → VCI) (%) | Mục tiêu dự báo (NPL). Ngoại lai 2,6% ô, 94% là của NVB. Lệch mạnh (skew 6,8): cần biến đổi nếu dùng mô hình tuyến tính. NVB (đang tái cơ cấu) lên 35% kéo lệch cả phân phối. NPL giảm ở quý 4 rồi tăng lại quý 1. |
| **credit_cost_ttm_pct** ⚠️ 🔍 | Chi phí dự phòng tín dụng TTM / dư nợ bình quân (%) | Ngoại lai 3,8% ô, 100% là của VPB. Lệch mạnh (skew 2,6): cần biến đổi nếu dùng mô hình tuyến tính. |
| **npl_group5_share_pct** | Nợ nhóm 5 / nợ xấu (nhóm 3–5) (%) | — |
| **provision_over_loans_percent** | Dự phòng / dư nợ (%) | — |
| **recovery_ttm_pct** | Thu hồi nợ đã xử lý TTM / dư nợ bình quân (%) | — |

#### M Quản trị

| Cột | Ý nghĩa | Ghi chú điều tra |
|---|---|---|
| **cir_ttm_pct** 🔍 | (TNHĐ − LN trước dự phòng) TTM / TNHĐ TTM (%) | Lệch mạnh (skew 3,1): cần biến đổi nếu dùng mô hình tuyến tính. |
| **deposit_insurance_to_deposits_pct** | Phí bảo hiểm tiền gửi TTM / tiền gửi bình quân (%) | — |
| **foreign_own_percent** | Sở hữu nước ngoài (ảnh chụp 09/09/2026) (%) | Không đổi theo thời gian: chỉ dùng để so sánh giữa các NH. |
| **opex_to_assets_ttm_pct** | Chi phí hoạt động TTM / tổng tài sản bình quân (%) | — |
| **staff_cost_share_pct** ⚠️ | Chi lương, phụ cấp TTM / chi phí hoạt động TTM (%) | Ngoại lai 6,3% ô, 76% là của ACB. |
| **state_own_percent** ⚠️ | Sở hữu nhà nước (ảnh chụp 09/09/2026) (%) | Không đổi theo thời gian: chỉ dùng để so sánh giữa các NH. Ngoại lai 18,2% ô, 25% là của BID. Phân ba nhóm rõ: 4 NH trên 40% (BID, VCB, CTG, MBB), 7 NH vài phần trăm, 11 NH bằng 0. |

#### E Thu nhập & chi phí vốn

| Cột | Ý nghĩa | Ghi chú điều tra |
|---|---|---|
| **cof_ttm_percent** 🎯 | Chi phí vốn TTM (VCI → MAS) (%) | Mục tiêu dự báo (COF). Đi theo lãi điều hành, trễ 1–2 quý (số riêng quý). 60% thay đổi là chung cả ngành. |
| **deposit_cost_implied_ttm_percent** ⚠️ | Trả lãi tiền gửi TTM / tiền gửi bình quân (suy ra) (%) | Cùng định nghĩa với COF trong cùng quý: chỉ dùng số quý trước. |
| **loan_yield_ttm_pct** | Thu lãi cho vay TTM / dư nợ bình quân (%) | — |
| **nim_ttm_percent** 🎯 | Biên lãi thuần TTM (VCI → MAS) (%) | Mục tiêu dự báo (NIM). Bị ép khi chi phí vốn tăng (trễ khoảng 2 quý). |
| **roa_ttm_percent** ⚠️ | ROA TTM (MAS) (%) | Cùng định nghĩa với ROE trong cùng quý: chỉ dùng số quý trước. Trùng thông tin với ROE (VIF ≈ 12): chỉ nên giữ một. |
| **roe_ttm_percent** 🎯 🔍 | ROE TTM (MAS) (%) | Mục tiêu dự báo (ROE). Lệch mạnh (skew −5,1): cần biến đổi nếu dùng mô hình tuyến tính. Gần như toàn bộ thay đổi là nhiễu riêng từng NH–quý. |
| **earning_asset_yield_ttm_percent** | Lợi suất tài sản sinh lãi TTM (MAS → suy ra) (%) | — |
| **fee_share_ttm_pct** | Lãi thuần dịch vụ TTM / TNHĐ TTM (%) | — |
| **noii_share_ttm_pct** | Thu nhập ngoài lãi TTM / TNHĐ TTM (%) | — |
| **npat_yoy_sym_pct** ⚠️ | Tăng trưởng đối xứng LNST TTM so cùng kỳ: 200(a−b)/(\|a\|+\|b\|) (%) | Cùng định nghĩa với ROE trong cùng quý: chỉ dùng số quý trước. Ngoại lai 4,7% ô, 71% là của NVB. |
| **toi_yoy_sym_pct** | Tăng trưởng đối xứng TNHĐ TTM so cùng kỳ: 200(a−b)/(\|a\|+\|b\|) (%) | — |
| **yield_cof_spread_pp** ⚠️ | Lợi suất TSSL TTM − chi phí vốn TTM (điểm %) | Cùng định nghĩa với NIM, COF trong cùng quý: chỉ dùng số quý trước. |

#### L Thanh khoản

| Cột | Ý nghĩa | Ghi chú điều tra |
|---|---|---|
| **casa_percent** 🎯 ★ | Tiền gửi không kỳ hạn / tổng tiền gửi (%) | Mục tiêu dự báo (CASA). Đi cùng thay đổi CASA quý sau (ρ = −0,28, cùng dấu ở hai nửa giai đoạn). So giữa các NH: CASA cao thì vốn rẻ (ρ = −0,95 với COF). Trong từng NH gần như không liên quan. Giảm ở quý 1. |
| **deposits_to_liabilities_pct** ⚠️ | Tiền gửi khách hàng / tổng nợ (%) | Cùng định nghĩa với LDR trong cùng quý: chỉ dùng số quý trước. |
| **gov_sbv_funding_pct** ⚠️ 🔍 | Nợ Chính phủ và NHNN / tổng nợ (%) | Ngoại lai 8,5% ô, 32% là của CTG. Lệch mạnh (skew 2,2): cần biến đổi nếu dùng mô hình tuyến tính. |
| **ldr_computed_percent** 🎯 | Dư nợ / tiền gửi khách hàng (%) | Mục tiêu dự báo (LDR). Có xu hướng (drift), không có mùa vụ. |
| **cfo_to_assets_ttm_pct** | Lưu chuyển tiền thuần HĐKD TTM / tổng tài sản bình quân (%) | — |
| **interbank_funding_pct** | Tiền gửi và vay TCTD khác / tổng nợ (%) | — |
| **liquid_assets_pct** | (Tiền + tiền gửi NHNN + tiền gửi/cho vay TCTD + CK kinh doanh) / tổng tài sản (%) | — |
| **loans_long_share_percent** | Cho vay dài hạn / dư nợ (%) | — |
| **loans_medium_share_percent** | Cho vay trung hạn / dư nợ (%) | — |
| **loans_over_assets_percent** ⚠️ | Dư nợ / tổng tài sản (%) | Cùng định nghĩa với LDR trong cùng quý: chỉ dùng số quý trước. |
| **loans_short_share_percent** | Cho vay ngắn hạn / dư nợ (%) | — |
| **paper_interest_share_pct** | Trả lãi giấy tờ có giá TTM / (trả lãi GTCG + trả lãi tiền gửi) TTM (%) | — |
| **papers_funding_pct** | Giấy tờ có giá phát hành / tổng nợ (%) | — |

#### S Nhạy cảm thị trường

| Cột | Ý nghĩa | Ghi chú điều tra |
|---|---|---|
| **ci_bond_to_assets_pct** | Trái phiếu TCTD khác / tổng tài sản (%) | — |
| **corp_bond_to_assets_pct** ⚠️ 🔍 | Trái phiếu TCKT trong nước / tổng tài sản (%) | Ngoại lai 4,4% ô, 52% là của TCB. Lệch mạnh (skew 2,7): cần biến đổi nếu dùng mô hình tuyến tính. |
| **gov_bond_share_invsec_pct** | Trái phiếu Chính phủ / chứng khoán đầu tư (%) | — |
| **invsec_to_assets_pct** | Chứng khoán đầu tư / tổng tài sản (%) | — |

### 4.2. Quy mô, tăng trưởng và động lượng

Ngân hàng lớn đến đâu, đang lớn nhanh hay chậm, và các chỉ tiêu vừa đổi bao nhiêu so với quý trước.

#### G Quy mô & tăng trưởng

| Cột | Ý nghĩa | Ghi chú điều tra |
|---|---|---|
| **loans_qoq_percent** ★ | Tăng trưởng dư nợ so quý trước (%) | Đi cùng thay đổi CASA quý sau (ρ = −0,28, cùng dấu ở hai nửa giai đoạn). Tín dụng dồn vào quý 4 (mùa vụ). |
| **loans_yoy_percent** ★ | Tăng trưởng dư nợ so cùng kỳ (%) | Đi cùng thay đổi ROE quý sau (ρ = +0,17, cùng dấu ở hai nửa giai đoạn). |
| **assets_yoy_pct** ★ | Tổng tài sản so cùng kỳ (%) | Đi cùng thay đổi ROE quý sau (ρ = +0,18, cùng dấu ở hai nửa giai đoạn). |
| **atm_per_branch** | Số ATM / số chi nhánh (lần) | Không đổi theo thời gian: chỉ dùng để so sánh giữa các NH. |
| **deposits_yoy_pct** | Tiền gửi khách hàng so cùng kỳ (%) | — |
| **equity_yoy_pct** 🔍 | Vốn chủ so cùng kỳ (%) | Lệch mạnh (skew 2,4): cần biến đổi nếu dùng mô hình tuyến tính. |
| **log_branches** | ln(1 + số chi nhánh/PGD trên OSM) (ln) | Không đổi theo thời gian: chỉ dùng để so sánh giữa các NH. |
| **log_total_assets** | ln(tổng tài sản, tỷ đồng) (ln) | Giả thuyết: NH càng lớn trả lãi tiền gửi càng thấp (ρ = −0,27 với lãi 12 tháng online, chưa có ý nghĩa với 22 NH). |
| **n_provinces_present** | Số tỉnh (34 tỉnh mới) có chi nhánh (số) | Không đổi theo thời gian: chỉ dùng để so sánh giữa các NH. |

#### D Động lượng

| Cột | Ý nghĩa | Ghi chú điều tra |
|---|---|---|
| **casa_chg_qoq_pp** ★ ⚠️ | Thay đổi casa_percent so quý trước (điểm %) | Đi cùng thay đổi CASA quý sau (ρ = −0,23, cùng dấu ở hai nửa giai đoạn). Cùng định nghĩa với CASA trong cùng quý: chỉ dùng số quý trước. |
| **cof_chg_qoq_pp** ★ ⚠️ | Thay đổi cof_ttm_percent so quý trước (điểm %) | Đi cùng thay đổi COF quý sau (ρ = +0,66, cùng dấu ở hai nửa giai đoạn). Cùng định nghĩa với COF trong cùng quý: chỉ dùng số quý trước. Ngoại lai 2,4% ô, 25% là của BVB. |
| **ldr_chg_qoq_pp** ★ ⚠️ | Thay đổi ldr_computed_percent so quý trước (điểm %) | Đi cùng thay đổi NIM quý sau (ρ = +0,15, cùng dấu ở hai nửa giai đoạn). Cùng định nghĩa với LDR trong cùng quý: chỉ dùng số quý trước. |
| **nim_chg_qoq_pp** ★ ⚠️ | Thay đổi nim_ttm_percent so quý trước (điểm %) | Đi cùng thay đổi NIM quý sau (ρ = +0,30, cùng dấu ở hai nửa giai đoạn). Cùng định nghĩa với NIM trong cùng quý: chỉ dùng số quý trước. |
| **npl_chg_qoq_pp** ★ ⚠️ | Thay đổi npl_percent so quý trước (điểm %) | Đi cùng thay đổi CASA quý sau (ρ = +0,23, cùng dấu ở hai nửa giai đoạn). Cùng định nghĩa với NPL trong cùng quý: chỉ dùng số quý trước. Ngoại lai 3,1% ô, 65% là của NVB. |
| **roe_chg_qoq_pp** ★ ⚠️ | Thay đổi roe_ttm_percent so quý trước (điểm %) | Đi cùng thay đổi ROE quý sau (ρ = +0,21, cùng dấu ở hai nửa giai đoạn). Cùng định nghĩa với ROE trong cùng quý: chỉ dùng số quý trước. |

### 4.3. Thị trường và vĩ mô

Tín hiệu từ giá cổ phiếu, và các biến chung cho mọi ngân hàng trong cùng quý.

#### T Thị trường

| Cột | Ý nghĩa | Ghi chú điều tra |
|---|---|---|
| **log_adtv** | ln(giá trị giao dịch bình quân phiên, đồng) (ln) | — |
| **pe** ★ ⚠️ 🔍 | P/E (MAS) (lần) | Đi cùng thay đổi COF quý sau (ρ = −0,25, cùng dấu ở hai nửa giai đoạn). Ngoại lai 5,4% ô, 58% là của NVB. Lệch mạnh (skew 20,1): cần biến đổi nếu dùng mô hình tuyến tính. |
| **beta** | Beta (MAS) (lần) | — |
| **dividend_yield** 🔍 | Tỷ suất cổ tức (MAS) (%) | Lệch mạnh (skew 4,0): cần biến đổi nếu dùng mô hình tuyến tính. |
| **excess_ret_q_pct** 🔍 | Lợi suất cổ phiếu − lợi suất VN-Index trong quý (%) | Lệch mạnh (skew 2,6): cần biến đổi nếu dùng mô hình tuyến tính. |
| **pb** ★ | P/B (MAS) (lần) | Đi cùng thay đổi COF quý sau (ρ = −0,22, cùng dấu ở hai nửa giai đoạn). |
| **stock_ret_q_pct** | Lợi suất cổ phiếu trong quý (%) | — |
| **stock_vol_q_pct** | Độ lệch chuẩn lợi suất log ngày × √252 (%) | — |

#### V Vĩ mô

| Cột | Ý nghĩa | Ghi chú điều tra |
|---|---|---|
| **overnight_sbv_rate_pct** ★ | Lãi suất cho vay qua dem NHNN có hiệu lực cuối quý (%) | Đi cùng thay đổi COF quý sau (ρ = +0,37, cùng dấu ở hai nửa giai đoạn). Chung cho mọi NH (vĩ mô): chỉ giải thích phần biến động chung theo quý. |
| **refinancing_rate_pct** ★ | Lãi suất tai cap von NHNN có hiệu lực cuối quý (%) | Đi cùng thay đổi COF quý sau (ρ = +0,47, cùng dấu ở hai nửa giai đoạn). Chung cho mọi NH (vĩ mô): chỉ giải thích phần biến động chung theo quý. Chỉ đổi 7 lần từ 2019, nên ít điểm để ước lượng độ trễ. |
| **system_credit_growth_asof_pct** ★ ⚠️ | Tăng trưởng tín dụng toàn hệ thống năm gần nhất (%) | Đi cùng thay đổi COF quý sau (ρ = +0,48, cùng dấu ở hai nửa giai đoạn). Chung cho mọi NH (vĩ mô): chỉ giải thích phần biến động chung theo quý. Số năm gán cho các quý sau (as-of): đổi theo bậc mỗi năm. Ngoại lai 10,0% ô, 5% là của ACB. |
| **vnindex_ret_q_pct** ★ | Lợi suất VN-Index trong quý (%) | Đi cùng thay đổi COF quý sau (ρ = −0,28, cùng dấu ở hai nửa giai đoạn). Chung cho mọi NH (vĩ mô): chỉ giải thích phần biến động chung theo quý. |
| **cpi_yoy_q_pct** | CPI so cùng kỳ, trung bình 3 tháng (%) | Chung cho mọi NH (vĩ mô): chỉ giải thích phần biến động chung theo quý. |
| **gold_index_yoy_q_pct** ⚠️ | Chỉ số giá vàng so cùng kỳ, trung bình 3 tháng (%) | Chung cho mọi NH (vĩ mô): chỉ giải thích phần biến động chung theo quý. Thiếu 30% ô. Chỉ số giá vàng (NSO) thiếu 9 quý nằm giữa 2021-Q2 và 2024-Q1. |
| **usd_index_yoy_q_pct** | Chỉ số giá USD so cùng kỳ, trung bình 3 tháng (%) | Chung cho mọi NH (vĩ mô): chỉ giải thích phần biến động chung theo quý. |
| **vnindex_vol_q_pct** | Biến động VN-Index (năm hóa) (%) | Chung cho mọi NH (vĩ mô): chỉ giải thích phần biến động chung theo quý. |

### 4.4. Chính sách tín dụng và khách hàng số

Tham gia gói tín dụng chính sách, cơ cấu dư nợ theo ngành, và mức độ người dùng tương tác với app.

#### P Chính sách & cơ cấu tín dụng

| Cột | Ý nghĩa | Ghi chú điều tra |
|---|---|---|
| **agri_share_asof_pct** | Dư nợ nông, lâm, thủy sản / dư nợ theo ngành (%) | Số năm gán cho các quý sau (as-of): đổi theo bậc mỗi năm. |
| **disaster_branch_share_pct** | % chi nhánh nằm ở tỉnh bị thiên tai trong cửa sổ hiệu lực (Yagi 9/2024–12/2025; bão 10–12 7/2025–) (%) | — |
| **ind_share_C_asof_pct** ⚠️ | Tỷ trọng dư nợ ngành công nghiệp chế biến (BCTC kiểm toán) (%) | Số năm gán cho các quý sau (as-of): đổi theo bậc mỗi năm. Thiếu 21% ô. Dư nợ theo ngành chép tay từ BCTC năm, còn 33 NH–năm chưa nhập. |
| **ind_share_F_asof_pct** ⚠️ 🔍 | Tỷ trọng dư nợ ngành xây dựng (BCTC kiểm toán) (%) | Số năm gán cho các quý sau (as-of): đổi theo bậc mỗi năm. Thiếu 19% ô. Ngoại lai 3,7% ô, 100% là của NVB. Lệch mạnh (skew 2,7): cần biến đổi nếu dùng mô hình tuyến tính. Dư nợ theo ngành chép tay từ BCTC năm, còn 33 NH–năm chưa nhập. |
| **n_programs_active** ⚠️ 🔍 | Số chương trình tín dụng chính sách đang tham gia (số) | Ngoại lai 5,6% ô, 11% là của BID. Lệch mạnh (skew 2,6): cần biến đổi nếu dùng mô hình tuyến tính. |
| **prog_hatang_500k** | = 1 nếu NH có trong danh sách HATANG_500K và quý kết thúc sau ngày danh sách (0/1) | — |
| **prog_lts_15k** | = 1 nếu NH có trong danh sách LTS_15K và quý kết thúc sau ngày danh sách (0/1) | — |
| **prog_nlts_100k** | = 1 nếu NH có trong danh sách NLTS_100K và quý kết thúc sau ngày danh sách (0/1) | — |
| **prog_nlts_185k** | = 1 nếu NH có trong danh sách NLTS_185K và quý kết thúc sau ngày danh sách (0/1) | — |
| **prog_noxh_145k** | = 1 nếu NH có trong danh sách NOXH_145K và quý kết thúc sau ngày danh sách (0/1) | — |
| **prog_noxh_u35** | = 1 nếu NH có trong danh sách NOXH_U35 và quý kết thúc sau ngày danh sách (0/1) | — |

#### K Khách hàng số

| Cột | Ý nghĩa | Ghi chú điều tra |
|---|---|---|
| **app_reviews_log** ⚠️ | ln(1 + số review trong quý) (ln) | Thiếu 17% ô. |

## 5. Insight ban đầu

**Tổng quan: dữ liệu đủ lưới và sạch trùng lặp.**
- 748 dòng = 22 NH × 34 quý, mỗi NH đúng 34 dòng. Khóa `(ticker, period)` là duy nhất, không có dòng trùng.
- Đây là lợi thế: như video nói, không phải xử lý trùng hay missing vội, nên các bước phía sau ổn định hơn.

**Kiểu dữ liệu: không có lỗi kiểu `TotalCharges`, nhưng có một cột ngày đang là chữ.**
- 88/91 cột là số (80 float, 8 int). Ba cột chữ: `ticker`, `period` (dạng `2024-Q3`) và `period_end`.
- `period_end` là ngày cuối quý nhưng lưu dạng chữ, nên đổi sang `datetime` khi làm chuỗi thời gian.
- Các bẫy về đơn vị và dấu (tỷ lệ lưu dạng thập phân, số âm, số 0 dùng thay ô trống) đã được xử lý từ trước ở `build_panel` và `validate`, nên không còn cột số nào bị lưu thành chữ.

**Thiếu 5,6% ô, và thiếu có lý do chứ không ngẫu nhiên.** 
- `ind_share_C_asof_pct`, `ind_share_F_asof_pct` (~26–28%): dư nợ theo ngành chép tay từ BCTC năm, còn 33 NH–năm chưa nhập.
- `gold_index_yoy_q_pct` (~26%): chỉ số giá vàng của NSO thiếu 9 quý nằm giữa 2021-Q2 và 2024-Q1.
- `car_asof_percent` (~24%): nhiều NH chỉ công bố CAR theo năm hoặc nửa năm.
- `app_reviews_log` (~22%): các quý có quá ít review, dồn vào 2018–2021.
- Các biến tăng trưởng so cùng kỳ (~12%): cần 4 quý lịch sử, nên 2018 trống theo định nghĩa.
- Từ 2019-Q1, tỷ lệ ô thiếu chỉ còn 2,3%.
- ⇒ Không điền bằng trung bình. Mô hình bắt đầu từ 2019-Q1, LightGBM để NaN.

**Mục tiêu: bài toán hồi quy có đuôi dày, thay cho mất cân bằng nhãn.**
- Cả 6 mục tiêu là số liên tục.
- Thay vào đó là đuôi rất dày: NPL lệch 7,1, độ nhọn 59, max 35,3% (NVB); ROE thấp nhất âm sâu, cũng ở NVB.
- ⇒ Cần hàm mất mát chịu được ngoại lai (MAE/Huber), và phải báo kết quả có/không có NVB.

**Chỉ có 22 thực thể.** 748 dòng, nhưng chỉ có 22 ngân hàng độc lập và 34 mốc thời gian.
- ⇒ Không chia train/test ngẫu nhiên: các quý của cùng một ngân hàng rất giống nhau, chia ngẫu nhiên sẽ rò rỉ.
- ⇒ Chia theo thời gian (walk-forward).

**Có đặc trưng rò rỉ cùng quý (⚠️).** Ví dụ `cof_chg_qoq_pp` được tính từ chính COF của quý đó. Dùng làm đầu vào thì chỉ được lấy số quý trước.

**Các file khác:**
- Ảnh chụp xếp hạng (22 × 90) chỉ thiếu vài ô có thể gọi tên: CAR của NVB, điểm app của BAB, dư nợ theo ngành của BVB, LPB, PGB, phí bảo hiểm tiền gửi của NAB, chi lương của SHB.
- File lãi suất theo ngày có 42% dòng không có `ticker`, vì đó là các ngân hàng ngoài 22 mã của đề tài: lọc bỏ chứ không phải điền.
- Cả hai panel và file lãi suất đều có cột ngày lưu dạng chữ.

## 6. Tổng kết và bước tiếp theo

**Kết luận EDA sơ bộ: dữ liệu đủ sạch để dựng baseline.**
- Không có trùng lặp.
- Thiếu có lý do và tập trung ở 2018.
- Kiểu dữ liệu đúng, trừ cột ngày cần đổi.
- Rủi ro chính không nằm ở độ sạch mà ở **đuôi dày (NVB)** và **cỡ mẫu nhỏ (22 NH)**.

Bước tiếp theo:
1. **EDA chi tiết** (`03_EDA_chi_tiet.ipynb`): phân phối thật của từng đặc trưng, ngoại lai có hợp logic không, tương quan, tương tác với mục tiêu, rò rỉ.
2. **Tiền xử lý**: đổi `period_end` sang datetime; chỉ dùng số quý trước cho các cột ⚠️ rò rỉ; winsorize hoặc Yeo-Johnson fit trên tập train (với mô hình tuyến tính).
3. **Baseline**: naive, naive có mùa vụ, drift; rồi mới so với mô hình học máy.